# Make music for the reader's theme library

Makes 30-second instrumental tracks with **MusicGen** (Meta's free music model). Claude turns them into seamless loops
(`make_loop.py`) and adds them to the reader's music library, where every chapter picks a **theme** for each stretch of story.

This run makes the themes the library is still missing, two takes each, so a theme sounds a little different when it comes back:

| Theme | Used for |
|---|---|
| `calm` (2nd take) | quiet moments, travel, the academy |
| `tension` (2nd take) | the build-up before something happens |
| `sad` | grief, loss, farewells |
| `eerie` | mystery, dungeons, the Relictombs |
| `triumph` | victory, a hard-won moment |
| `warm` | family, friends, comedy |

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Menu **Runtime → Run all**. About 15 minutes, most of it the first download of the model.
3. A zip called `music_takes.zip` downloads at the end (it is also saved to `Storyteller/music_takes` in your Google Drive). Send it to Claude.

To try different music, change the descriptions in Step 2 and run Steps 2 to 4 again. Every description says
*instrumental, no vocals*: lyrics distract from reading. MusicGen's model is free for non-commercial use, which fits a personal project.

### Step 1: load the music model (about 5 minutes the first time)

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

MODEL = "facebook/musicgen-large"   # best quality; "facebook/musicgen-medium" is faster and lighter
gpu = torch.cuda.is_available()
processor = AutoProcessor.from_pretrained(MODEL)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float16 if gpu else torch.float32)
model = model.to("cuda" if gpu else "cpu").eval()
RATE = model.config.audio_encoder.sampling_rate
print("Loaded on the", "graphics card" if gpu else "processor (very slow: choose the T4 GPU runtime)")

### Step 2: describe the music

In [ ]:
# One track per description. The name is the theme and the take: calm_2 is the second take of "calm".
# Already in the library: calm_1, tension_1, battle_1-2, battle_dark_1-2, climax_1-2.
TRACKS = {
    "calm_2": "peaceful fantasy village morning, acoustic guitar and soft flute over warm strings, gentle and nostalgic, "
              "85 bpm, instrumental, no vocals",
    "tension_2": "suspenseful cinematic underscore, low pulsing strings, ticking percussion, distant brass swells, "
                 "something is about to happen, 90 bpm, instrumental, no vocals",
    "sad_1": "sad emotional piano melody with soft strings, grief and loss, slow and tender, cinematic anime drama, "
             "70 bpm, instrumental, no vocals",
    "sad_2": "melancholic solo cello and piano, quiet rain atmosphere, heartbreaking farewell, slow, 65 bpm, instrumental, no vocals",
    "eerie_1": "dark eerie ambient, deep drones, distant metallic echoes, mysterious ancient ruins, unsettling, slow, "
               "instrumental, no vocals",
    "eerie_2": "dark synthwave mystery, slow pulsing analog synth bass, cold atmospheric pads, lonely and tense exploration, "
               "90 bpm, instrumental, no vocals",
    "triumph_1": "triumphant heroic orchestral theme, soaring brass fanfare, full strings and wordless choir, victory after a hard battle, "
                 "uplifting, 110 bpm, instrumental",
    "triumph_2": "uplifting epic post-rock build, shimmering guitars and strings, big steady drums, hope and resolve, "
                 "120 bpm, instrumental, no vocals",
    "warm_1": "lighthearted playful fantasy music, pizzicato strings, woodwinds and glockenspiel, cheerful slice of life, "
              "comedic, 110 bpm, instrumental, no vocals",
    "warm_2": "warm cozy family theme, acoustic guitar, soft piano and light strings, home and comfort, gentle, 90 bpm, "
              "instrumental, no vocals",
}
SECONDS = 30   # MusicGen makes up to about 30 seconds at a time

### Step 3: make the tracks (about a minute each)

In [ ]:
import numpy as np, soundfile as sf, subprocess, os
from IPython.display import Audio, display

os.makedirs("music_takes", exist_ok=True)
for n, (name, description) in enumerate(TRACKS.items()):
    torch.manual_seed(100 + n)  # new seeds: different music from earlier runs
    inputs = processor(text=[description], padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        audio = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(SECONDS * 50))
    x = audio[0, 0].float().cpu().numpy()

    # make it loop: blend the last 1.5 seconds into the start, so the end runs straight back into the beginning
    fade = int(1.5 * RATE)
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    loop = x[:-fade].copy()
    loop[:fade] = x[:fade] * ramp + x[-fade:] * (1 - ramp)

    # one fixed volume for the whole track (a time-varying loudness filter would break the loop seam)
    loop = loop * (0.89 / max(1e-6, float(np.abs(loop).max())))
    wav = f"music_takes/{name}.wav"
    sf.write(wav, loop, RATE)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav,
                    "-ar", "44100", "-c:a", "libmp3lame", "-b:a", "128k", wav[:-4] + ".mp3"], check=True)
    os.remove(wav)
    print(f"{name}: {len(loop) / RATE:.0f} s")
    display(Audio(wav[:-4] + ".mp3"))

### Step 4: download

In [ ]:
import os, shutil
from google.colab import files
!cd music_takes && zip -qr ../music_takes.zip .
# also keep a copy in Google Drive, next to the chapters
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Storyteller/music_takes', exist_ok=True)
    shutil.copy('music_takes.zip', '/content/drive/MyDrive/Storyteller/music_takes/music_takes.zip')
    print('Saved to Storyteller/music_takes in Google Drive')
except Exception as e:
    print('Not saved to Google Drive:', e)
files.download('music_takes.zip')